# Pipeline V5 - HackUDC Zara
**Mejoras V5 (sobre V4):**
- **Multi-crop training**: Usa TODOS los crops compatibles por producto (2-3x mas datos de entrenamiento)
- **Hard negative mining**: Cada 10 epochs, re-mina negativos dificiles con FAISS
- **Warmup + early stopping**: LR warmup 5 epochs + paciencia 12 + temperatura 0.05
- **Accessory popularity boost**: Zapatos/accesorios frecuentes boosteados x3.5 (data: top shoe reused 38x)
- **Category-aware timestamps**: Ventanas temporales distintas para ropa (<30d) vs accesorios (<120d)
- **Co-occurrence boost**: Top-8 co-ocurrencias con peso x7 para accesorios

**V4:** ViT-L-14-336, TTA flip, alpha-QE, DBA
**V3:** Zone crops feet/head, diversity enforcement, search_k=25

Ejecutar TODAS las celdas en orden.

In [ ]:
# CELDA 1: INSTALAR DEPENDENCIAS
!pip install -q open-clip-torch faiss-cpu ultralytics huggingface_hub

In [ ]:
# CELDA 2: IMPORTS Y CONFIG
import csv, re, json, sys, os, gc, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open_clip
import faiss
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import defaultdict, Counter
import torchvision.transforms.functional as TF

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# RUTAS
WORK_DIR = Path('/kaggle/working')
BASE_DIR = Path('/kaggle/input/datasets/kkjasdkjo/hackudc')
IMAGES_DIR = WORK_DIR / 'images'
BUNDLES_DIR = IMAGES_DIR / 'bundles'
PRODUCTS_DIR = IMAGES_DIR / 'products'
EMBEDDINGS_DIR = WORK_DIR / 'embeddings'
SUBMISSIONS_DIR = WORK_DIR / 'submissions'
CROPS_DIR = WORK_DIR / 'crops'
MODELS_DIR = WORK_DIR / 'models'

for d in [IMAGES_DIR, BUNDLES_DIR, PRODUCTS_DIR, EMBEDDINGS_DIR, SUBMISSIONS_DIR, CROPS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# V4: Higher resolution CLIP model
CLIP_MODEL_NAME = 'ViT-L-14-336'  # 336px instead of 224px
CLIP_DIM = 768
USE_YOLO = True
USE_TTA = True  # Test-Time Augmentation (horizontal flip)
NUM_EPOCHS = 50

# HUMAN LABELS PATHS
HUMAN_LABELS_PATH = Path('/kaggle/input/datasets/kkjasdkjo/human-labels/human_labels.csv')
HUMAN_NEGATIVES_PATH = Path('/kaggle/input/datasets/kkjasdkjo/human-labels/human_negatives.csv')
print(f'Human labels: {HUMAN_LABELS_PATH.exists()}')
print(f'Human negatives: {HUMAN_NEGATIVES_PATH.exists()}')
print(f'CLIP model: {CLIP_MODEL_NAME}')
print(f'TTA: {USE_TTA}')
print('Config OK')

In [ ]:
# CELDA 3: CONSTANTES Y MAPEOS YOLO
YOLO_TO_CATALOG = {
    'short_sleeved_shirt': ['T-SHIRT', 'POLO SHIRT', 'BABY T-SHIRT', 'TOPS AND OTHERS'],
    'long_sleeved_shirt': ['SHIRT', 'SWEATER', 'CARDIGAN', 'SWEATSHIRT', 'OVERSHIRT',
                           'BABY SHIRT', 'BABY SWEATER', 'BABY CARDIGAN'],
    'short_sleeved_outwear': ['BLAZER', 'WAISTCOAT'],
    'long_sleeved_outwear': ['WIND-JACKET', 'COAT', 'ANORAK', 'BLAZER', 'OVERSHIRT',
                              'TRENCH RAINCOAT', 'BABY JACKET/COAT', 'BABY WIND-JACKET'],
    'vest': ['WAISTCOAT', 'BODYSUIT', 'TOPS AND OTHERS'],
    'sling': ['TOPS AND OTHERS', 'BODYSUIT', 'DRESS'],
    'shorts': ['BERMUDA', 'SHORTS', 'BABY BERMUDAS'],
    'trousers': ['TROUSERS', 'LEGGINGS', 'BABY TROUSERS', 'BABY LEGGINGS'],
    'skirt': ['SKIRT', 'BABY SKIRT'],
    'short_sleeved_dress': ['DRESS', 'BABY DRESS', 'OVERALL'],
    'long_sleeved_dress': ['DRESS', 'BABY DRESS', 'OVERALL'],
    'vest_dress': ['DRESS', 'BABY DRESS'],
    'sling_dress': ['DRESS', 'BABY DRESS'],
}

EXTRA_GROUPS = {
    'footwear': ['SHOES', 'FLAT SHOES', 'SANDAL', 'HEELED SHOES', 'MOCCASINS', 'SPORT SHOES',
                 'RUNNING SHOES', 'TRAINERS', 'ANKLE BOOT', 'FLAT ANKLE BOOT', 'HEELED ANKLE BOOT',
                 'FLAT BOOT', 'HEELED BOOT', 'HIGH TOPS', 'BOOT', 'RAIN BOOT', 'ATHLETIC FOOTWEAR',
                 'SPORTY SANDAL', 'SNEAKERS', 'BABY SHOES'],
    'bags': ['HAND BAG-RUCKSACK', 'PURSE WALLET'],
    'headwear': ['HAT', 'GLASSES', 'BABY BONNET', 'HAT/HEADBAND'],
    'accessories': ['BELT', 'IMIT JEWELLER', 'SCARF', 'SOCKS', 'TIE', 'ACCESSORIES',
                    'GLOVES', 'SHAWL/FOULARD', 'PANTY/UNDERPANT', 'STOCKINGS-TIGHTS', 'BABY SOCKS'],
}

ALL_GROUPS = {}
ALL_GROUPS.update(YOLO_TO_CATALOG)
ALL_GROUPS.update(EXTRA_GROUPS)

CAT_TO_GROUP = {}
for group_name, cats in ALL_GROUPS.items():
    for cat in cats:
        if cat not in CAT_TO_GROUP:
            CAT_TO_GROUP[cat] = group_name

ZONE_CROPS = {
    'upper': [(0.05, 0.50, 0.05, 0.95), (0.10, 0.45, 0.10, 0.90)],
    'lower': [(0.45, 0.85, 0.05, 0.95), (0.40, 0.80, 0.10, 0.90)],
    'feet':  [(0.75, 1.00, 0.0, 1.0), (0.80, 0.98, 0.10, 0.90)],
    'full':  [(0.05, 0.90, 0.05, 0.95)],
    'head':  [(0.00, 0.18, 0.15, 0.85)],
}

ZONE_TO_CATALOG = {
    'upper': ['T-SHIRT', 'SHIRT', 'SWEATER', 'BLAZER', 'WIND-JACKET', 'SWEATSHIRT',
              'CARDIGAN', 'COAT', 'ANORAK', 'WAISTCOAT', 'OVERSHIRT', 'TOPS AND OTHERS',
              'POLO SHIRT', 'BODYSUIT', 'TRENCH RAINCOAT',
              'BABY T-SHIRT', 'BABY SWEATER', 'BABY SHIRT', 'BABY JACKET/COAT',
              'BABY WIND-JACKET', 'BABY CARDIGAN'],
    'lower': ['TROUSERS', 'SKIRT', 'BERMUDA', 'SHORTS', 'LEGGINGS',
              'BABY TROUSERS', 'BABY BERMUDAS', 'BABY SKIRT', 'BABY LEGGINGS'],
    'feet':  EXTRA_GROUPS['footwear'],
    'full':  ['DRESS', 'OVERALL', 'BABY DRESS', 'BABY OUTFIT', 'SWIMSUIT', 'BIB OVERALL'],
    'head':  EXTRA_GROUPS['headwear'],
}

# ZONE_TO_GROUP: maps zone crop labels to FAISS group names for targeted search
ZONE_TO_GROUP = {
    'feet': ['footwear'],
    'head': ['headwear'],
    'upper': ['short_sleeved_shirt', 'long_sleeved_shirt', 'short_sleeved_outwear',
              'long_sleeved_outwear', 'vest', 'sling'],
    'lower': ['trousers', 'shorts', 'skirt'],
    'full': ['short_sleeved_dress', 'long_sleeved_dress', 'vest_dress', 'sling_dress'],
}

# Category groups for diversity enforcement
DIVERSITY_GROUPS = {
    'TOPS': {'T-SHIRT', 'SHIRT', 'SWEATER', 'CARDIGAN', 'SWEATSHIRT', 'POLO SHIRT',
             'TOPS AND OTHERS', 'BODYSUIT', 'OVERSHIRT', 'BABY T-SHIRT', 'BABY SHIRT',
             'BABY SWEATER', 'BABY CARDIGAN'},
    'BOTTOMS': {'TROUSERS', 'BERMUDA', 'SHORTS', 'LEGGINGS', 'BABY TROUSERS',
                'BABY BERMUDAS', 'BABY LEGGINGS'},
    'SHOES': set(EXTRA_GROUPS['footwear']),
    'OUTERWEAR': {'BLAZER', 'COAT', 'ANORAK', 'WIND-JACKET', 'WAISTCOAT',
                  'TRENCH RAINCOAT', 'BABY JACKET/COAT', 'BABY WIND-JACKET'},
    'DRESS': {'DRESS', 'OVERALL', 'BABY DRESS'},
    'SKIRT': {'SKIRT', 'BABY SKIRT'},
}

print('Constantes OK')

In [ ]:
# CELDA 4: FUNCIONES UTILIDAD
def load_csv(path):
    with open(path) as f:
        return list(csv.DictReader(f))

def extract_sku(url):
    for pattern in [r'/(\d{8,15}(?:-\d+)?)-[pe]', r'/(T\d{8,15}(?:-\d+)?)-[pe]',
                    r'/(M\d{8,15}(?:-\d+)?)-[pe]']:
        match = re.search(pattern, str(url))
        if match:
            return match.group(1)
    return None

def extract_ts(url):
    match = re.search(r'ts=(\d+)', str(url))
    return int(match.group(1)) if match else None

def get_bundle_path(bid):
    return BUNDLES_DIR / f'{bid}.jpg'

def crop_zone(img, y1_pct, y2_pct, x1_pct, x2_pct):
    w, h = img.size
    return img.crop((int(w * x1_pct), int(h * y1_pct), int(w * x2_pct), int(h * y2_pct)))

def crop_bbox(img, bbox, padding=0.1):
    w, h = img.size
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, x1 - bw * padding)
    y1 = max(0, y1 - bh * padding)
    x2 = min(w, x2 + bw * padding)
    y2 = min(h, y2 + bh * padding)
    return img.crop((int(x1), int(y1), int(x2), int(y2)))

print('Funciones OK')

In [ ]:
# CELDA 5: CLIP + TTA
def load_clip():
    print(f'Cargando CLIP {CLIP_MODEL_NAME} en {DEVICE}...')
    model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME, pretrained='openai', device=DEVICE
    )
    model.eval()
    if DEVICE == 'cuda':
        model = model.half()
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
    return model, preprocess, tokenizer

def embed_pil(model, preprocess, img):
    """Embed a single PIL image with optional TTA (horizontal flip averaging)."""
    tensor = preprocess(img).unsqueeze(0).to(DEVICE)
    if DEVICE == 'cuda':
        tensor = tensor.half()
    with torch.no_grad():
        feat = model.encode_image(tensor)
        feat = feat / feat.norm(dim=-1, keepdim=True)

        if USE_TTA:
            # Horizontal flip TTA
            img_flip = TF.hflip(img)
            tensor_flip = preprocess(img_flip).unsqueeze(0).to(DEVICE)
            if DEVICE == 'cuda':
                tensor_flip = tensor_flip.half()
            feat_flip = model.encode_image(tensor_flip)
            feat_flip = feat_flip / feat_flip.norm(dim=-1, keepdim=True)
            # Average and re-normalize
            feat = (feat + feat_flip) / 2
            feat = feat / feat.norm(dim=-1, keepdim=True)

    return feat.cpu().float().numpy().flatten()

def embed_batch_from_dir(model, preprocess, image_dir, ids, batch_size=32):
    """Embed batch of images with optional TTA. Reduced batch size for 336px."""
    all_embs = []
    for i in tqdm(range(0, len(ids), batch_size), desc='Embeddings'):
        batch_ids = ids[i:i + batch_size]
        images_orig = []
        images_flip = []
        for img_id in batch_ids:
            try:
                pil_img = Image.open(image_dir / f'{img_id}.jpg').convert('RGB')
                img = preprocess(pil_img)
                images_orig.append(img)
                if USE_TTA:
                    images_flip.append(preprocess(TF.hflip(pil_img)))
            except Exception:
                blank = Image.new('RGB', (336, 336))
                img = preprocess(blank)
                images_orig.append(img)
                if USE_TTA:
                    images_flip.append(img)

        batch = torch.stack(images_orig).to(DEVICE)
        if DEVICE == 'cuda':
            batch = batch.half()
        with torch.no_grad():
            feats = model.encode_image(batch)
            feats = feats / feats.norm(dim=-1, keepdim=True)

            if USE_TTA and images_flip:
                batch_flip = torch.stack(images_flip).to(DEVICE)
                if DEVICE == 'cuda':
                    batch_flip = batch_flip.half()
                feats_flip = model.encode_image(batch_flip)
                feats_flip = feats_flip / feats_flip.norm(dim=-1, keepdim=True)
                feats = (feats + feats_flip) / 2
                feats = feats / feats.norm(dim=-1, keepdim=True)

        all_embs.append(feats.cpu().float().numpy())
    return np.vstack(all_embs)

print('CLIP funciones OK (con TTA)' if USE_TTA else 'CLIP funciones OK')

In [ ]:
# CELDA 6: YOLO
def load_yolo_model():
    from huggingface_hub import hf_hub_download
    from ultralytics import YOLO
    print('Cargando YOLOv8 DeepFashion2...')
    path = hf_hub_download('Bingsu/adetailer', 'deepfashion2_yolov8s-seg.pt')
    return YOLO(path)

def detect_garments(yolo_model, image_path, conf=0.15):
    results = yolo_model(str(image_path), verbose=False, conf=conf)
    detections = []
    if results and results[0].boxes is not None:
        r = results[0]
        for i in range(len(r.boxes)):
            label = r.names[int(r.boxes.cls[i])]
            bbox = r.boxes.xyxy[i].cpu().numpy()
            conf_score = float(r.boxes.conf[i])
            detections.append({'label': label, 'bbox': bbox, 'conf': conf_score})
    return detections

print('YOLO funciones OK')

In [ ]:
# CELDA 7: MODELO - ProjectionHead + Contrastive Training V5
# V5: Hard negative mining, warmup, lower temperature, early stopping

class ProjectionHead(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = dim * 2
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x) + x, dim=-1)


class ContrastiveDataset(Dataset):
    def __init__(self, crop_embs, pos_product_embs, all_product_embs, neg_indices_per_sample, num_negatives=31):
        self.crop_embs = crop_embs
        self.pos_product_embs = pos_product_embs
        self.all_product_embs = all_product_embs
        self.neg_indices = neg_indices_per_sample
        self.num_negatives = num_negatives

    def __len__(self):
        return len(self.crop_embs)

    def __getitem__(self, idx):
        crop = torch.FloatTensor(self.crop_embs[idx])
        pos = torch.FloatTensor(self.pos_product_embs[idx])
        neg_pool = self.neg_indices[idx]
        if len(neg_pool) > self.num_negatives:
            chosen = random.sample(list(neg_pool), self.num_negatives)
        else:
            chosen = list(neg_pool)
            while len(chosen) < self.num_negatives:
                chosen.append(random.choice(chosen))
        negs = torch.FloatTensor(self.all_product_embs[chosen])
        return crop, pos, negs


def mine_hard_negatives(model, crop_embs, pos_prod_indices, product_embeddings, k=128):
    """Use current model to find hard negatives via FAISS search."""
    model.eval()
    prod_embs = product_embeddings.copy().astype(np.float32)
    faiss.normalize_L2(prod_embs)
    index_hn = faiss.IndexFlatIP(prod_embs.shape[1])
    index_hn.add(prod_embs)

    all_hard_negs = []
    batch_size = 512
    with torch.no_grad():
        for i in range(0, len(crop_embs), batch_size):
            batch_embs = crop_embs[i:i+batch_size]
            batch = torch.FloatTensor(batch_embs).to(DEVICE)
            projected = model(batch).cpu().numpy().astype(np.float32)
            faiss.normalize_L2(projected)
            _, indices = index_hn.search(projected, k)

            for j in range(len(batch_embs)):
                pos_idx = pos_prod_indices[i + j]
                hard = [int(idx) for idx in indices[j] if idx != pos_idx]
                all_hard_negs.append(hard[:k-1])

    return all_hard_negs


def train_projection(train_data, val_data, product_embeddings, dim, num_epochs=50, lr=3e-4):
    model = ProjectionHead(dim).to(DEVICE)
    crop_embs_train, pos_embs_train, neg_indices_train, pos_prod_idx_train = train_data
    crop_embs_val, pos_embs_val, neg_indices_val = val_data[0], val_data[1], val_data[2]

    train_dataset = ContrastiveDataset(crop_embs_train, pos_embs_train, product_embeddings, neg_indices_train)
    val_dataset = ContrastiveDataset(crop_embs_val, pos_embs_val, product_embeddings, neg_indices_val)

    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # V5: Warmup + Cosine annealing
    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs)
        return max(0.01, 0.5 * (1 + np.cos(np.pi * progress)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    temperature = 0.05  # V5: Lower temperature for sharper discrimination
    best_val_loss = float('inf')
    best_state = None
    patience = 12
    patience_counter = 0
    mining_interval = 10  # V5: Re-mine hard negatives every N epochs

    for epoch in range(num_epochs):
        # V5: Online hard negative mining
        if epoch > 0 and epoch % mining_interval == 0 and pos_prod_idx_train:
            print(f'  [Epoch {epoch}] Mining hard negatives...')
            hard_negs = mine_hard_negatives(model, crop_embs_train, pos_prod_idx_train, product_embeddings)
            train_dataset = ContrastiveDataset(crop_embs_train, pos_embs_train, product_embeddings, hard_negs)
            train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)

        model.train()
        train_losses = []
        for crop, pos, negs in train_loader:
            crop, pos, negs = crop.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)
            projected = model(crop)
            pos_sim = (projected * pos).sum(-1, keepdim=True)
            neg_sim = torch.bmm(negs, projected.unsqueeze(-1)).squeeze(-1)
            logits = torch.cat([pos_sim, neg_sim], dim=-1) / temperature
            labels = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
            loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        model.eval()
        val_losses = []
        correct = 0
        total = 0
        with torch.no_grad():
            for crop, pos, negs in val_loader:
                crop, pos, negs = crop.to(DEVICE), pos.to(DEVICE), negs.to(DEVICE)
                projected = model(crop)
                pos_sim = (projected * pos).sum(-1, keepdim=True)
                neg_sim = torch.bmm(negs, projected.unsqueeze(-1)).squeeze(-1)
                logits = torch.cat([pos_sim, neg_sim], dim=-1) / temperature
                labels = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
                loss = F.cross_entropy(logits, labels)
                val_losses.append(loss.item())
                correct += (logits.argmax(dim=-1) == 0).sum().item()
                total += logits.size(0)

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        val_acc = correct / total if total > 0 else 0

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0 or epoch == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch+1:3d}/{num_epochs}: train={train_loss:.4f} val={val_loss:.4f} acc={val_acc:.3f} lr={lr_now:.6f}')

        # V5: Early stopping
        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.4f})')
            break

    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    return model

print('Modelo V5 OK (multi-crop, hard mining, warmup, early stopping)')

In [ ]:
# CELDA 8: COPIAR IMAGENES DESDE DATASET
import shutil

# Buscar imagenes en los inputs de Kaggle
src = Path('/kaggle/input/hackudc-images/images')
if src.exists():
    for subdir in ['bundles', 'products']:
        src_dir = src / subdir
        dst_dir = IMAGES_DIR / subdir
        if src_dir.exists() and not any(dst_dir.iterdir()):
            print(f'Copiando {subdir}...')
            shutil.copytree(str(src_dir), str(dst_dir), dirs_exist_ok=True)
            print(f'  {len(os.listdir(dst_dir))} archivos')
    print('Imagenes copiadas!')
else:
    print(f'No se encontro {src}')
    print('Buscando en otros paths...')
    for root, dirs, files in os.walk('/kaggle/input'):
        jpg_count = sum(1 for f in files if f.endswith('.jpg'))
        if jpg_count > 100:
            print(f'  {root}: {jpg_count} jpgs')

print(f'Bundles: {len(list(BUNDLES_DIR.glob("*.jpg")))} imagenes')
print(f'Products: {len(list(PRODUCTS_DIR.glob("*.jpg")))} imagenes')

In [ ]:
# CELDA 9: CARGAR DATOS
print('[FASE 1] Cargando datos...')
bundles = load_csv(BASE_DIR / 'bundles_dataset.csv')
products = load_csv(BASE_DIR / 'product_dataset.csv')
train_pairs = load_csv(BASE_DIR / 'bundles_product_match_train.csv')
test_rows = load_csv(BASE_DIR / 'bundles_product_match_test.csv')

bundle_section = {b['bundle_asset_id']: b['bundle_id_section'] for b in bundles}
bundle_url_map = {b['bundle_asset_id']: b['bundle_image_url'] for b in bundles}
product_desc = {p['product_asset_id']: p['product_description'] for p in products}
product_url_map = {p['product_asset_id']: p['product_image_url'] for p in products}
product_ids_all = [p['product_asset_id'] for p in products]

# SKU indexes
sku_to_products = defaultdict(list)
sku_prefix_to_products = {n: defaultdict(list) for n in [4, 5, 6, 7, 8]}
for p in products:
    sku = extract_sku(p['product_image_url'])
    if sku:
        sku_to_products[sku].append(p['product_asset_id'])
        for n in sku_prefix_to_products:
            if len(sku) >= n:
                sku_prefix_to_products[n][sku[:n]].append(p['product_asset_id'])

# Section constraints
cat_sections = defaultdict(set)
for row in train_pairs:
    sec = bundle_section.get(row['bundle_asset_id'])
    desc = product_desc.get(row['product_asset_id'])
    if sec and desc:
        cat_sections[desc].add(sec)

# Training bundle -> products
train_bundle_products = defaultdict(set)
for row in train_pairs:
    train_bundle_products[row['bundle_asset_id']].add(row['product_asset_id'])

# Co-occurrence
cooccurrence = defaultdict(Counter)
for bid, pids in train_bundle_products.items():
    for p1 in pids:
        for p2 in pids:
            if p1 != p2:
                cooccurrence[p1][p2] += 1

# Popularity per section
section_product_freq = defaultdict(Counter)
for row in train_pairs:
    sec = bundle_section.get(row['bundle_asset_id'])
    if sec:
        section_product_freq[sec][row['product_asset_id']] += 1

# Timestamps
bundle_ts = {}
for b in bundles:
    ts = extract_ts(b['bundle_image_url'])
    if ts:
        bundle_ts[b['bundle_asset_id']] = ts
product_ts = {}
for p in products:
    ts = extract_ts(p['product_image_url'])
    if ts:
        product_ts[p['product_asset_id']] = ts

# === CARGAR HUMAN LABELS ===
human_bids = set()
human_labels_direct = defaultdict(set)  # CLAVE: para inyeccion directa en predicciones
human_negatives_direct = defaultdict(set)  # Para penalizar negativos

if HUMAN_LABELS_PATH.exists():
    human_rows = load_csv(HUMAN_LABELS_PATH)
    human_count = 0
    for row in human_rows:
        bid = row['bundle_asset_id']
        pid = row['product_asset_id']
        if pid:
            train_bundle_products[bid].add(pid)
            human_labels_direct[bid].add(pid)
            human_bids.add(bid)
            human_count += 1
    print(f'  Human labels: {human_count} positivos para {len(human_bids)} bundles')
    print(f'  >>> INYECCION DIRECTA activada para {len(human_labels_direct)} bundles <<<')
else:
    print('  Sin human labels')

if HUMAN_NEGATIVES_PATH.exists():
    neg_rows = load_csv(HUMAN_NEGATIVES_PATH)
    neg_count = 0
    for row in neg_rows:
        bid = row['bundle_asset_id']
        pid = row['product_asset_id']
        if pid:
            human_negatives_direct[bid].add(pid)
            neg_count += 1
    print(f'  Human negatives: {neg_count} negativos para {len(human_negatives_direct)} bundles')

# Train/val split (human bundles SIEMPRE a training)
original_bids = [bid for bid in list(train_bundle_products.keys()) if bid not in human_bids]
random.seed(42)
random.shuffle(original_bids)
split = int(0.8 * len(original_bids))
train_bids = original_bids[:split] + list(human_bids)
val_bids = original_bids[split:]
print(f'  Train: {len(train_bids)} ({len(human_bids)} human), Val: {len(val_bids)}')
print(f'  Productos: {len(product_ids_all)}')
print('Datos OK')

In [ ]:
# CELDA 10: EMBEDDINGS CLIP
print('[FASE 2] Embeddings CLIP...')
clip_model, clip_preprocess, clip_tokenizer = load_clip()

model_tag = CLIP_MODEL_NAME.replace('-', '').replace('/', '').lower()

# Product embeddings
prod_emb_path = EMBEDDINGS_DIR / f'products_{model_tag}.npy'
prod_ids_path = EMBEDDINGS_DIR / f'product_ids_{model_tag}.json'

if prod_emb_path.exists() and prod_ids_path.exists():
    print('  Cargando embeddings productos...')
    product_embeddings = np.load(prod_emb_path)
    with open(prod_ids_path) as f:
        product_ids = json.load(f)
else:
    product_ids = product_ids_all
    print(f'  Generando embeddings de {len(product_ids)} productos...')
    product_embeddings = embed_batch_from_dir(clip_model, clip_preprocess, PRODUCTS_DIR, product_ids)
    np.save(prod_emb_path, product_embeddings)
    with open(prod_ids_path, 'w') as f:
        json.dump(product_ids, f)

pid_to_idx = {pid: i for i, pid in enumerate(product_ids)}
pid_to_emb_idx = pid_to_idx  # alias
print(f'  {len(product_ids)} productos, dim={product_embeddings.shape[1]}')

# Bundle embeddings
all_bundle_bids = list(train_bundle_products.keys())
bundle_emb_path = EMBEDDINGS_DIR / f'bundles_{model_tag}.npy'
bundle_ids_path = EMBEDDINGS_DIR / f'bundle_ids_{model_tag}.json'

if bundle_emb_path.exists() and bundle_ids_path.exists():
    print('  Cargando embeddings bundles...')
    all_bundle_embs = np.load(bundle_emb_path)
    with open(bundle_ids_path) as f:
        all_bundle_bids_ordered = json.load(f)
else:
    all_bundle_bids_ordered = all_bundle_bids
    print(f'  Generando embeddings de {len(all_bundle_bids)} bundles...')
    all_embs = []
    for i in tqdm(range(0, len(all_bundle_bids_ordered), 64), desc='Bundles'):
        batch_ids = all_bundle_bids_ordered[i:i + 64]
        images = []
        for bid in batch_ids:
            try:
                img = clip_preprocess(Image.open(get_bundle_path(bid)).convert('RGB'))
            except Exception:
                img = clip_preprocess(Image.new('RGB', (224, 224)))
            images.append(img)
        batch = torch.stack(images).to(DEVICE)
        if DEVICE == 'cuda':
            batch = batch.half()
        with torch.no_grad():
            feats = clip_model.encode_image(batch)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        all_embs.append(feats.cpu().float().numpy())
    all_bundle_embs = np.vstack(all_embs)
    np.save(bundle_emb_path, all_bundle_embs)
    with open(bundle_ids_path, 'w') as f:
        json.dump(all_bundle_bids_ordered, f)

bid_to_emb_idx = {bid: i for i, bid in enumerate(all_bundle_bids_ordered)}

# Asegurar float32
all_bundle_embs = all_bundle_embs.astype(np.float32)
product_embeddings = product_embeddings.astype(np.float32)

print('Embeddings OK')

In [ ]:
# CELDA 11: YOLO DETECTION + CROP EMBEDDINGS + ZONE SUPPLEMENT
print('[FASE 3] Deteccion de prendas...')
# Cache key v2 for zone crops (feet + head only)
crops_cache_path = EMBEDDINGS_DIR / f'crops_data_{model_tag}_v2.pkl'

if crops_cache_path.exists():
    print('  Cargando crops cacheados (v2 con zone crops)...')
    with open(crops_cache_path, 'rb') as f:
        crops_data = pickle.load(f)
else:
    yolo_model = None
    if USE_YOLO:
        try:
            yolo_model = load_yolo_model()
        except Exception as e:
            print(f'  YOLO no disponible: {e}')

    crops_data = {}
    for bid in tqdm(all_bundle_bids_ordered, desc='Detecting crops'):
        img_path = get_bundle_path(bid)
        try:
            bundle_img = Image.open(img_path).convert('RGB')
        except Exception:
            crops_data[bid] = []
            continue

        crops_for_bundle = []
        if yolo_model:
            detections = detect_garments(yolo_model, img_path)
            for det in detections:
                cropped = crop_bbox(bundle_img, det['bbox'], padding=0.08)
                emb = embed_pil(clip_model, clip_preprocess, cropped)
                crops_for_bundle.append({'label': det['label'], 'embedding': emb, 'conf': det['conf']})

            # Zone crops for feet/head (YOLO can't detect shoes/hats)
            for zone_name in ['feet', 'head']:
                for coords in ZONE_CROPS[zone_name]:
                    cropped = crop_zone(bundle_img, *coords)
                    emb = embed_pil(clip_model, clip_preprocess, cropped)
                    crops_for_bundle.append({'label': zone_name, 'embedding': emb, 'conf': 0.4})
        else:
            for zone_name, zone_coords_list in ZONE_CROPS.items():
                for coords in zone_coords_list:
                    cropped = crop_zone(bundle_img, *coords)
                    emb = embed_pil(clip_model, clip_preprocess, cropped)
                    crops_for_bundle.append({'label': zone_name, 'embedding': emb, 'conf': 0.5})

        full_emb = embed_pil(clip_model, clip_preprocess, bundle_img)
        crops_for_bundle.append({'label': '_full_', 'embedding': full_emb, 'conf': 1.0})
        crops_data[bid] = crops_for_bundle

    if yolo_model:
        del yolo_model
        torch.cuda.empty_cache()
        gc.collect()

    with open(crops_cache_path, 'wb') as f:
        pickle.dump(crops_data, f)

# Asegurar float32 en crops
for bid in crops_data:
    for crop in crops_data[bid]:
        crop['embedding'] = crop['embedding'].astype(np.float32)

# Count crop types
crop_type_counts = Counter()
for bid in crops_data:
    for crop in crops_data[bid]:
        crop_type_counts[crop['label']] += 1
print(f'  Crops para {len(crops_data)} bundles')
print(f'  Crop types: {dict(crop_type_counts.most_common(15))}')
print('Crops OK')

In [ ]:
# CELDA 12: CROPS PARA BUNDLES HUMANOS (test bundles anotados)
if human_bids:
    test_crops_need = [bid for bid in human_bids if bid not in crops_data]
    if test_crops_need:
        print(f'Generando crops para {len(test_crops_need)} human bundles...')
        yolo_model = load_yolo_model()
        for bid in tqdm(test_crops_need, desc='Human crops'):
            img_path = get_bundle_path(bid)
            try:
                bundle_img = Image.open(img_path).convert('RGB')
            except Exception:
                crops_data[bid] = []
                continue
            crops_for_bundle = []
            detections = detect_garments(yolo_model, img_path)
            for det in detections:
                cropped = crop_bbox(bundle_img, det['bbox'], padding=0.08)
                emb = embed_pil(clip_model, clip_preprocess, cropped)
                crops_for_bundle.append({'label': det['label'], 'embedding': emb.astype(np.float32), 'conf': det['conf']})
            # Zone crops for feet/head
            for zone_name in ['feet', 'head']:
                for coords in ZONE_CROPS[zone_name]:
                    cropped = crop_zone(bundle_img, *coords)
                    emb = embed_pil(clip_model, clip_preprocess, cropped)
                    crops_for_bundle.append({'label': zone_name, 'embedding': emb.astype(np.float32), 'conf': 0.4})
            full_emb = embed_pil(clip_model, clip_preprocess, bundle_img)
            crops_for_bundle.append({'label': '_full_', 'embedding': full_emb.astype(np.float32), 'conf': 1.0})
            crops_data[bid] = crops_for_bundle
        del yolo_model
        torch.cuda.empty_cache()
        gc.collect()
    print(f'Human bundles con crops: {len([b for b in human_bids if b in crops_data])}')
else:
    print('Sin human bundles')

In [ ]:
# CELDA 13: CREAR PARES DE ENTRENAMIENTO - V5: Multi-crop training
print('[FASE 4] Creando pares de entrenamiento (V5 multi-crop)...')

neg_pool_cache = {}
section_pool_cache = {}

for sec in ['1', '2', '3']:
    sec_indices = []
    for i, pid in enumerate(product_ids):
        desc = product_desc.get(pid, '')
        if desc not in cat_sections or sec in cat_sections[desc]:
            sec_indices.append(i)
    section_pool_cache[sec] = sec_indices

    cat_indices = defaultdict(list)
    for i in sec_indices:
        desc = product_desc.get(product_ids[i], '')
        cat_indices[desc].append(i)
    for cat, indices in cat_indices.items():
        neg_pool_cache[(cat, sec)] = indices

def create_pairs(bids_list):
    crop_embs_list = []
    pos_embs_list = []
    neg_indices_list = []
    pos_prod_idx_list = []  # V5: track product indices for hard negative mining

    for bid in bids_list:
        sec = bundle_section.get(bid, '1')
        matched_products = train_bundle_products.get(bid, set())
        if bid not in crops_data or not crops_data[bid]:
            continue

        for pid in matched_products:
            if pid not in pid_to_idx:
                continue
            product_cat = product_desc.get(pid, '')
            product_emb_idx = pid_to_idx[pid]
            product_emb = product_embeddings[product_emb_idx]

            # Build negative pool (same as before)
            key = (product_cat, sec)
            hard_neg_indices = [i for i in neg_pool_cache.get(key, []) if product_ids[i] != pid]
            if len(hard_neg_indices) < 10:
                hard_neg_indices = [i for i in section_pool_cache.get(sec, []) if product_ids[i] != pid][:500]
            if not hard_neg_indices:
                continue

            # V5: Use ALL compatible crops (not just best)
            # More training pairs = better generalization
            used_any = False
            for crop_info in crops_data[bid]:
                crop_label = crop_info['label']
                if crop_label == '_full_':
                    continue  # Skip full - too noisy for specific matching
                compatible = False
                if crop_label in YOLO_TO_CATALOG:
                    compatible = product_cat in YOLO_TO_CATALOG[crop_label]
                elif crop_label in ZONE_TO_CATALOG:
                    compatible = product_cat in ZONE_TO_CATALOG[crop_label]
                if compatible:
                    sim = np.dot(crop_info['embedding'], product_emb)
                    if sim > 0.12:  # Only use crops with reasonable similarity
                        crop_embs_list.append(crop_info['embedding'])
                        pos_embs_list.append(product_emb)
                        neg_indices_list.append(hard_neg_indices)
                        pos_prod_idx_list.append(product_emb_idx)
                        used_any = True

            # Fallback to full image if no compatible crop found
            if not used_any:
                for crop_info in crops_data[bid]:
                    if crop_info['label'] == '_full_':
                        crop_embs_list.append(crop_info['embedding'])
                        pos_embs_list.append(product_emb)
                        neg_indices_list.append(hard_neg_indices)
                        pos_prod_idx_list.append(product_emb_idx)
                        break

    return (
        np.array(crop_embs_list) if crop_embs_list else np.zeros((0, CLIP_DIM)),
        np.array(pos_embs_list) if pos_embs_list else np.zeros((0, CLIP_DIM)),
        neg_indices_list,
        pos_prod_idx_list,
    )

train_data = create_pairs(train_bids)
print(f'  {len(train_data[0])} pares de entrenamiento (multi-crop)')
val_data = create_pairs(val_bids)
print(f'  {len(val_data[0])} pares de validacion')
print('Pares OK')

In [ ]:
# CELDA 14: ENTRENAR PROJECTION HEAD
print(f'[FASE 5] Entrenando projection head ({NUM_EPOCHS} epochs)...')
if len(train_data[0]) == 0:
    print('ERROR: No hay pares de entrenamiento!')
    projection = None
else:
    projection = train_projection(
        train_data, val_data, product_embeddings, CLIP_DIM,
        num_epochs=NUM_EPOCHS, lr=3e-4
    )
    torch.save(projection.state_dict(), MODELS_DIR / f'projection_{model_tag}.pt')
    print('Entrenamiento completado!')

In [ ]:
# CELDA 15: CONSTRUIR INDICES FAISS + DBA (Database-side Augmentation)
print('[FASE 6] Construyendo indices FAISS...')

# product_section mapping
product_section = {}
for pid in product_ids:
    desc = product_desc.get(pid, '')
    if desc in cat_sections:
        product_section[pid] = list(cat_sections[desc])[0]
    else:
        product_section[pid] = '1'

# === DBA: Database-side Augmentation ===
# Smooth each product embedding with its k nearest neighbors
# This reduces noise and improves retrieval
def apply_dba(embeddings, k=5, alpha=3.0):
    """Database-side augmentation: replace each embedding with
    a weighted average of itself and its k nearest neighbors."""
    n, d = embeddings.shape
    embs = embeddings.copy().astype(np.float32)
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(d)
    index.add(embs)
    sims, indices = index.search(embs, k + 1)  # +1 for self

    augmented = np.zeros_like(embs)
    for i in range(n):
        # Self weight = 1, neighbor weights = sim^alpha
        weights = np.ones(k + 1, dtype=np.float32)
        weights[1:] = np.maximum(sims[i, 1:k+1], 0) ** alpha
        weights /= weights.sum()
        neighbor_embs = embs[indices[i, :k+1]]
        augmented[i] = (weights[:, None] * neighbor_embs).sum(0)

    faiss.normalize_L2(augmented)
    return augmented

print('  Applying DBA to product embeddings...')
product_embs_dba = apply_dba(product_embeddings, k=5, alpha=3.0)

# Group indices (using DBA embeddings)
group_indices = {}
for group_name, catalog_cats in ALL_GROUPS.items():
    cats_set = set(catalog_cats)
    for sec in ['1', '2', '3']:
        idxs, pids = [], []
        for pid in product_ids:
            desc = product_desc.get(pid, '')
            if desc not in cats_set:
                continue
            if desc in cat_sections and sec not in cat_sections[desc]:
                continue
            if pid in pid_to_idx:
                idxs.append(pid_to_idx[pid])
                pids.append(pid)
        if idxs:
            embs = product_embs_dba[idxs].copy().astype(np.float32)
            faiss.normalize_L2(embs)
            index = faiss.IndexFlatIP(embs.shape[1])
            index.add(embs)
            group_indices[(group_name, sec)] = (index, pids)

# Section-wide indices (using DBA embeddings)
section_indices = {}
section_pids_map = {}
for sec in ['1', '2', '3']:
    idxs, pids = [], []
    for pid in product_ids:
        desc = product_desc.get(pid, '')
        if desc in cat_sections and sec not in cat_sections[desc]:
            continue
        if pid in pid_to_idx:
            idxs.append(pid_to_idx[pid])
            pids.append(pid)
    embs = product_embs_dba[idxs].copy().astype(np.float32)
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    section_indices[sec] = index
    section_pids_map[sec] = pids

# Bundle-to-bundle (ALL training bundles)
all_section_b2b = {}
for sec in ['1', '2', '3']:
    idxs, bids_sec = [], []
    for i, bid in enumerate(all_bundle_bids_ordered):
        if bundle_section.get(bid) == sec:
            idxs.append(i)
            bids_sec.append(bid)
    if idxs:
        embs = all_bundle_embs[idxs].copy().astype(np.float32)
        faiss.normalize_L2(embs)
        index = faiss.IndexFlatIP(embs.shape[1])
        index.add(embs)
        all_section_b2b[sec] = (index, bids_sec)

print(f'  Group indices: {len(group_indices)}')
print(f'  Section indices: {len(section_indices)}')
print(f'  B2B indices: {len(all_section_b2b)}')
print(f'  DBA applied: k=5, alpha=3.0')
print('Indices OK')

In [ ]:
# CELDA 16: PREDICT_BUNDLE V5 - Multi-crop + Hard mining + Data-driven scoring
# V5 improvements:
# - Stronger accessory popularity (data shows top shoes appear 38x)
# - Category-aware temporal windows (clothing <30d, accessories <120d)
# - Increased projection weight (better model from V5 training)
# - Smarter co-occurrence boosting

# Pre-compute accessory categories for popularity boosting
ACCESSORY_CATS = {'SHOES', 'MOCCASINS', 'SANDAL', 'SNEAKERS', 'FLAT SHOES', 'HEELED SHOES',
                  'ANKLE BOOT', 'HEELED BOOT', 'TRAINERS', 'RAIN BOOT', 'BABY SHOES',
                  'SOCKS', 'BELT', 'GLASSES', 'HAT/HEADBAND', 'SCARF',
                  'HAND BAG-RUCKSACK', 'IMIT JEWELLER'}
CLOTHING_CATS = {'T-SHIRT', 'SHIRT', 'SWEATER', 'TROUSERS', 'SHORTS', 'BERMUDA',
                 'BLAZER', 'COAT', 'WIND-JACKET', 'CARDIGAN', 'SWEATSHIRT', 'OVERSHIRT',
                 'DRESS', 'SKIRT', 'OVERALL', 'POLO SHIRT', 'VEST'}

def predict_bundle(bid, b2b_indices, use_projection=True):
    sec = bundle_section.get(bid, '1')
    burl = bundle_url_map.get(bid, '')
    bsku = extract_sku(burl)
    bts = bundle_ts.get(bid)
    candidates = {}

    # SIGNAL 0: Human labels (if available)
    if bid in human_labels_direct:
        for pid in human_labels_direct[bid]:
            candidates[pid] = candidates.get(pid, 0) + 500

    # SIGNAL 1: SKU exact
    if bsku and bsku in sku_to_products:
        for pid in sku_to_products[bsku]:
            candidates[pid] = candidates.get(pid, 0) + 200

    # SIGNAL 2: SKU prefix (including first-8 for color variants)
    prefix_scores = {8: 100, 7: 70, 6: 45, 5: 28, 4: 15}  # V5: boosted first-8
    if bsku:
        for plen, pscore in prefix_scores.items():
            if len(bsku) >= plen:
                prefix = bsku[:plen]
                for pid in sku_prefix_to_products[plen].get(prefix, []):
                    desc = product_desc.get(pid, '')
                    if desc in cat_sections and sec not in cat_sections[desc]:
                        continue
                    old = candidates.get(pid, 0)
                    candidates[pid] = max(old, pscore) if old < 100 else old + pscore

    # SIGNAL 3: Bundle-to-bundle (top-40)
    bundle_emb = None
    if bid in bid_to_emb_idx:
        bundle_emb = all_bundle_embs[bid_to_emb_idx[bid]].reshape(1, -1).copy().astype(np.float32)
    elif bid in crops_data:
        for c in crops_data[bid]:
            if c['label'] == '_full_':
                bundle_emb = c['embedding'].reshape(1, -1).copy().astype(np.float32)
                break

    if bundle_emb is not None and sec in b2b_indices:
        faiss.normalize_L2(bundle_emb)
        b2b_index, b2b_bids = b2b_indices[sec]
        k = min(40, b2b_index.ntotal)
        sim_scores, sim_idx = b2b_index.search(bundle_emb, k)
        for j in range(k):
            similar_bid = b2b_bids[sim_idx[0][j]]
            if similar_bid == bid:
                continue
            sim = float(sim_scores[0][j])
            if sim < 0.3:
                continue
            for pid in train_bundle_products.get(similar_bid, set()):
                bonus = sim * 35
                candidates[pid] = candidates.get(pid, 0) + bonus
                # V5: Stronger co-occurrence for accessories
                for copid, count in cooccurrence[pid].most_common(8):  # V5: top-8
                    if copid in pid_to_idx:
                        cop_desc = product_desc.get(copid, '')
                        co_weight = 7 if cop_desc in ACCESSORY_CATS else 5  # V5: boost accessories
                        candidates[copid] = candidates.get(copid, 0) + count * sim * co_weight

    # SIGNAL 4: Projection head - handles zone crops via ZONE_TO_GROUP
    if bid in crops_data and projection is not None and use_projection:
        for crop_info in crops_data[bid]:
            crop_label = crop_info['label']
            if crop_label == '_full_':
                continue
            crop_emb = torch.FloatTensor(crop_info['embedding']).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                projected = projection(crop_emb).cpu().numpy().astype(np.float32)
            faiss.normalize_L2(projected)

            if crop_label in YOLO_TO_CATALOG:
                groups_to_search = [crop_label]
            elif crop_label in ZONE_TO_GROUP:
                groups_to_search = ZONE_TO_GROUP[crop_label]
            else:
                groups_to_search = []

            for group in groups_to_search:
                key = (group, sec)
                if key in group_indices:
                    index, gpids = group_indices[key]
                    search_k = min(25, index.ntotal)
                    scores, indices = index.search(projected, search_k)
                    for j2 in range(search_k):
                        pid = gpids[indices[0][j2]]
                        clip_score = float(scores[0][j2])
                        conf = crop_info.get('conf', 0.5)
                        bonus = clip_score * (1 + conf) * 25  # V5: 25 from 20
                        candidates[pid] = candidates.get(pid, 0) + bonus

    # SIGNAL 5: Raw CLIP - handles zone crops via ZONE_TO_GROUP
    if bid in crops_data:
        for crop_info in crops_data[bid]:
            crop_label = crop_info['label']
            if crop_label == '_full_':
                continue
            crop_emb = crop_info['embedding'].reshape(1, -1).copy().astype(np.float32)
            faiss.normalize_L2(crop_emb)

            if crop_label in YOLO_TO_CATALOG:
                groups_to_search = [crop_label]
            elif crop_label in ZONE_TO_GROUP:
                groups_to_search = ZONE_TO_GROUP[crop_label]
            else:
                groups_to_search = []

            for group in groups_to_search:
                key = (group, sec)
                if key in group_indices:
                    index, gpids = group_indices[key]
                    search_k = min(25, index.ntotal)
                    scores, indices = index.search(crop_emb, search_k)
                    for j2 in range(search_k):
                        pid = gpids[indices[0][j2]]
                        clip_score = float(scores[0][j2])
                        bonus = clip_score * 12
                        candidates[pid] = candidates.get(pid, 0) + bonus

    # SIGNAL 6: Popularity - V5: category-aware weighting
    # Accessories are reused 5-38x more than clothing
    for pid, freq in section_product_freq[sec].most_common(80):  # V5: 80 from 50
        desc = product_desc.get(pid, '')
        if desc in ACCESSORY_CATS:
            candidates[pid] = candidates.get(pid, 0) + freq * 3.5  # V5: strong accessory prior
        else:
            candidates[pid] = candidates.get(pid, 0) + freq * 1.5

    # SIGNAL 7: Timestamp - V5: category-aware windows
    if bts:
        for pid in list(candidates.keys()):
            if pid in product_ts:
                diff_days = abs(bts - product_ts[pid]) / (1000 * 86400)
                desc = product_desc.get(pid, '')
                is_accessory = desc in ACCESSORY_CATS

                if diff_days < 7:
                    candidates[pid] = candidates[pid] * 1.25 + 6
                elif diff_days < 30:
                    candidates[pid] = candidates[pid] * 1.12 + 3
                elif diff_days < 90:
                    candidates[pid] = candidates[pid] * 1.05 + 1
                elif diff_days < 120 and is_accessory:
                    pass  # Accessories have longer shelf life
                elif diff_days > 180:
                    penalty = 0.6 if not is_accessory else 0.8
                    candidates[pid] *= penalty

    # SIGNAL 8: Section-wide search (full image fallback)
    if bundle_emb is not None and sec in section_indices:
        scores, indices = section_indices[sec].search(bundle_emb, 80)
        spids = section_pids_map[sec]
        for j in range(min(80, len(spids))):
            pid = spids[indices[0][j]]
            base_score = float(scores[0][j]) * 2.5
            if pid not in candidates:
                candidates[pid] = base_score
            elif candidates[pid] < 50:
                candidates[pid] += base_score * 0.5

    # PENALIZATION: Human negatives
    if bid in human_negatives_direct:
        for pid in human_negatives_direct[bid]:
            if pid in candidates:
                candidates[pid] = min(candidates[pid] * 0.1, 5)

    # ALPHA-QUERY EXPANSION (alpha-QE)
    if bundle_emb is not None and sec in section_indices:
        top_pids = sorted(candidates.items(), key=lambda x: -x[1])[:10]
        expansion_embs = []
        expansion_scores = []
        for pid, sc in top_pids:
            if sc > 20 and pid in pid_to_idx:
                expansion_embs.append(product_embeddings[pid_to_idx[pid]])
                expansion_scores.append(sc)
        if expansion_embs:
            alpha = 3.0
            weights = np.array(expansion_scores, dtype=np.float32)
            weights = weights ** (1.0 / alpha)
            weights = weights / weights.sum()
            expanded = bundle_emb.flatten().copy()
            for emb, w in zip(expansion_embs, weights):
                expanded += w * emb
            expanded = expanded.reshape(1, -1).astype(np.float32)
            faiss.normalize_L2(expanded)
            scores2, indices2 = section_indices[sec].search(expanded, 40)
            spids = section_pids_map[sec]
            for j in range(min(40, len(spids))):
                pid = spids[indices2[0][j]]
                qe_score = float(scores2[0][j]) * 4.0
                if pid not in candidates:
                    candidates[pid] = qe_score
                elif candidates[pid] < 40:
                    candidates[pid] += qe_score * 0.4

    # ================================================================
    # CATEGORY DIVERSITY ENFORCEMENT
    # ================================================================
    sorted_c = sorted(candidates.items(), key=lambda x: -x[1])
    top15 = sorted_c[:15]
    remaining = sorted_c[15:60]  # V5: larger pool for swapping

    represented = set()
    for pid, score in top15:
        desc = product_desc.get(pid, '')
        for group_name, descs in DIVERSITY_GROUPS.items():
            if desc in descs:
                represented.add(group_name)

    expected = {'TOPS', 'BOTTOMS', 'SHOES'}
    if 'DRESS' in represented:
        expected = {'SHOES'}

    missing = expected - represented
    if missing and remaining:
        for group_name in list(missing):
            best_swap = None
            for pid, score in remaining:
                desc = product_desc.get(pid, '')
                if desc in DIVERSITY_GROUPS.get(group_name, set()):
                    best_swap = (pid, score)
                    break
            if best_swap:
                replace_idx = len(top15) - 1
                for idx in range(len(top15) - 1, -1, -1):
                    p, s = top15[idx]
                    p_desc = product_desc.get(p, '')
                    p_cats = [g for g, descs in DIVERSITY_GROUPS.items() if p_desc in descs]
                    if not p_cats:
                        replace_idx = idx
                        break
                    p_cat = p_cats[0]
                    cat_count = sum(1 for pp, _ in top15 if product_desc.get(pp, '') in DIVERSITY_GROUPS.get(p_cat, set()))
                    if cat_count > 1:
                        replace_idx = idx
                        break
                top15[replace_idx] = best_swap
                top15.sort(key=lambda x: -x[1])
                missing.discard(group_name)

    return top15

print('predict_bundle V5 cargado!')
print(f'  Accessory popularity boost: ACTIVE')
print(f'  Category-aware timestamps: ACTIVE')
print(f'  Projection weight: 25 (up from 20)')
print(f'  Co-occurrence: top-8 with accessory boost')
print(f'  Human labels: {len(human_labels_direct)} bundles')
print(f'  Human negatives: {len(human_negatives_direct)} bundles')

In [ ]:
# CELDA 17: VALIDACION
print('[FASE 6b] Validacion...')

# B2B solo con train (no val) para validacion justa
train_section_b2b = {}
for sec in ['1', '2', '3']:
    idxs, bids_sec = [], []
    train_set = set(train_bids)
    for i, bid in enumerate(all_bundle_bids_ordered):
        if bid in train_set and bundle_section.get(bid) == sec:
            idxs.append(i)
            bids_sec.append(bid)
    if idxs:
        embs = all_bundle_embs[idxs].copy().astype(np.float32)
        faiss.normalize_L2(embs)
        index = faiss.IndexFlatIP(embs.shape[1])
        index.add(embs)
        train_section_b2b[sec] = (index, bids_sec)

val_hits = 0
val_total = 0
for bid in tqdm(val_bids, desc='Validating'):
    true_products = train_bundle_products.get(bid, set())
    predicted = predict_bundle(bid, train_section_b2b, use_projection=(projection is not None))
    predicted_pids = set(pid for pid, score in predicted)
    hits = len(predicted_pids & true_products)
    val_hits += hits
    val_total += len(true_products)

val_recall = val_hits / val_total if val_total > 0 else 0
print(f'\n  VALIDACION: recall@15 = {val_recall:.4f} ({val_hits}/{val_total})')

In [ ]:
# CELDA 18: INFERENCIA EN TEST
print('[FASE 7] Inferencia en test...')
test_bids = list(set(row['bundle_asset_id'] for row in test_rows))
print(f'  Test bundles: {len(test_bids)}')
print(f'  Con human labels: {len([b for b in test_bids if b in human_labels_direct])}')
print(f'  Con human negatives: {len([b for b in test_bids if b in human_negatives_direct])}')

# Crops para test bundles - WITH zone crops for feet/head
test_crops_need = [bid for bid in test_bids if bid not in crops_data]
if test_crops_need:
    print(f'  Generando crops para {len(test_crops_need)} test bundles...')
    yolo_model = load_yolo_model()
    for bid in tqdm(test_crops_need, desc='Test crops'):
        img_path = get_bundle_path(bid)
        try:
            bundle_img = Image.open(img_path).convert('RGB')
        except Exception:
            crops_data[bid] = []
            continue
        crops_for_bundle = []
        if yolo_model:
            detections = detect_garments(yolo_model, img_path)
            for det in detections:
                cropped = crop_bbox(bundle_img, det['bbox'], padding=0.08)
                emb = embed_pil(clip_model, clip_preprocess, cropped)
                crops_for_bundle.append({'label': det['label'], 'embedding': emb.astype(np.float32), 'conf': det['conf']})
            # Zone crops for feet/head (YOLO can't detect shoes/hats)
            for zone_name in ['feet', 'head']:
                for coords in ZONE_CROPS[zone_name]:
                    cropped = crop_zone(bundle_img, *coords)
                    emb = embed_pil(clip_model, clip_preprocess, cropped)
                    crops_for_bundle.append({'label': zone_name, 'embedding': emb.astype(np.float32), 'conf': 0.4})
        full_emb = embed_pil(clip_model, clip_preprocess, bundle_img)
        crops_for_bundle.append({'label': '_full_', 'embedding': full_emb.astype(np.float32), 'conf': 1.0})
        crops_data[bid] = crops_for_bundle
    del yolo_model
    torch.cuda.empty_cache()
    gc.collect()

# Test bundle embeddings
for bid in test_bids:
    if bid not in bid_to_emb_idx and bid in crops_data:
        for c in crops_data[bid]:
            if c['label'] == '_full_':
                idx = len(all_bundle_bids_ordered)
                all_bundle_bids_ordered.append(bid)
                all_bundle_embs = np.vstack([all_bundle_embs, c['embedding'].reshape(1, -1).astype(np.float32)])
                bid_to_emb_idx[bid] = idx
                break

# Predict
results = []
human_injected_count = 0
human_products_injected = 0
diversity_swaps = 0

for bid in tqdm(test_bids, desc='PREDICTING'):
    predicted = predict_bundle(bid, all_section_b2b, use_projection=(projection is not None))
    pids_predicted = [pid for pid, score in predicted]

    # Track diversity enforcement
    descs_in_pred = [product_desc.get(pid, '') for pid in pids_predicted]
    has_shoes = any(d in DIVERSITY_GROUPS.get('SHOES', set()) for d in descs_in_pred)
    if has_shoes:
        diversity_swaps += 1

    # Count human label injections
    if bid in human_labels_direct:
        human_injected_count += 1
        injected_in_top15 = len(set(pids_predicted) & human_labels_direct[bid])
        human_products_injected += injected_in_top15

    # Fill to 15
    if len(pids_predicted) < 15:
        sec = bundle_section.get(bid, '1')
        seen = set(pids_predicted)
        spids = section_pids_map.get(sec, product_ids)
        for pid in spids:
            if pid not in seen:
                pids_predicted.append(pid)
                seen.add(pid)
                if len(pids_predicted) >= 15:
                    break

    for pid in pids_predicted[:15]:
        results.append({'bundle_asset_id': bid, 'product_asset_id': pid})

# Save
out_path = SUBMISSIONS_DIR / f'submission_v5_{model_tag}.csv'
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['bundle_asset_id', 'product_asset_id'])
    writer.writeheader()
    writer.writerows(results)

print(f'\n{"=" * 60}')
print(f'  SUBMISSION V5: {out_path}')
print(f'  Filas: {len(results)}')
print(f'  Bundles: {len(test_bids)}')
print(f'  Val recall@15: {val_recall:.4f}')
print(f'  Bundles con shoes en top-15: {diversity_swaps}/{len(test_bids)}')
print(f'  Human labels inyectados: {human_injected_count}/{len(test_bids)} bundles')
print(f'  Productos humanos en top-15: {human_products_injected}')
print(f'{"=" * 60}')
print(f'\nMEJORAS V5:')
print(f'  [V5] Multi-crop training + hard negative mining')
print(f'  [V5] Accessory popularity boost + category-aware timestamps')
print(f'  [V3] Zone crops feet/head + search_k=25 + diversity enforcement')
print(f'  [V4] ViT-L-14-336 (336px resolution)')
print(f'  [V4] TTA horizontal flip')
print(f'  [V4] Alpha-Query Expansion')
print(f'  [V4] Database-side Augmentation (DBA)')

from IPython.display import FileLink
display(FileLink(str(out_path)))